# GitHub Archive — Exploratory Data Analysis

**Schema:** `databricks_certification_ml.github_archive`

This notebook explores four tables derived from the [GitHub Archive](https://www.gharchive.org/) public dataset:

| Table | Description |
|---|---|
| `github_events` | Core event stream — PRs, issues, pushes, comments, reviews, releases, etc. (54 columns) |
| `commits` | Individual commit references with author and push metadata (8 columns) |
| `file_changes` | File-level diffs — additions, deletions, changed file counts per event (15 columns) |
| `line_changes` | Line-level diffs — hunks, positions, and inline review comments (13 columns) |

---

In [0]:
%sql
USE CATALOG XXXX;
USE SCHEMA github_archive;

In [0]:
%sql
-- Row counts for every table in the schema
SELECT 'github_events' AS table_name, COUNT(*) AS row_count FROM github_events
UNION ALL
SELECT 'commits',       COUNT(*) FROM commits
UNION ALL
SELECT 'file_changes',  COUNT(*) FROM file_changes
UNION ALL
SELECT 'line_changes',  COUNT(*) FROM line_changes
ORDER BY row_count DESC

table_name,row_count
github_events,50000000
file_changes,3580991
commits,3276183
line_changes,876966


In [0]:
%sql
-- Earliest and latest timestamps across each table
SELECT 'github_events' AS table_name,
       MIN(created_at) AS earliest,
       MAX(created_at) AS latest,
       DATEDIFF(MAX(created_at), MIN(created_at)) AS span_days
FROM github_events
UNION ALL
SELECT 'commits', MIN(created_at), MAX(created_at),
       DATEDIFF(MAX(created_at), MIN(created_at))
FROM commits
UNION ALL
SELECT 'file_changes', MIN(created_at), MAX(created_at),
       DATEDIFF(MAX(created_at), MIN(created_at))
FROM file_changes
UNION ALL
SELECT 'line_changes', MIN(created_at), MAX(created_at),
       DATEDIFF(MAX(created_at), MIN(created_at))
FROM line_changes
ORDER BY table_name

table_name,earliest,latest,span_days
commits,2011-10-08T05:35:04.000Z,2022-04-16T22:59:49.000Z,3843
file_changes,2011-02-12T05:22:11.000Z,2022-03-12T02:59:36.000Z,4046
github_events,2011-02-12T00:01:31.000Z,2022-04-16T22:59:52.000Z,4081
line_changes,2013-02-22T20:41:25.000Z,2022-03-12T02:50:24.000Z,3305


## 1. Event Type Distribution

The `github_events` table captures many event types. Let's see what activity looks like across event types, time, repos, and contributors.

In [0]:
%sql
-- Volume of each GitHub event type
SELECT
    event_type,
    COUNT(*)                          AS event_count,
    COUNT(DISTINCT actor_login)       AS unique_actors,
    COUNT(DISTINCT repo_name)         AS unique_repos,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM github_events
GROUP BY event_type
ORDER BY event_count DESC

event_type,event_count,unique_actors,unique_repos,pct_of_total
PushEvent,24483456,395150,1551132,49.0
CreateEvent,7215802,482142,2370298,14.4
PullRequestEvent,3606321,202924,242606,7.2
WatchEvent,3399680,1245341,243929,6.8
IssueCommentEvent,3293175,285501,160394,6.6
CommitCommentEvent,2736880,351707,504966,5.5
IssuesEvent,2129907,322076,117610,4.3
DeleteEvent,1130496,37150,90129,2.3
ForkEvent,1122304,739098,150096,2.2
PullRequestReviewCommentEvent,881979,30569,16317,1.8


In [0]:
%sql
-- Monthly event volume — top 5 event types vs everything else
WITH top5 AS (
    SELECT event_type
    FROM github_events
    GROUP BY event_type
    ORDER BY COUNT(*) DESC
    LIMIT 5
)
SELECT
    DATE_TRUNC('month', ge.created_at) AS month,
    COALESCE(t.event_type, 'Other')    AS event_type,
    COUNT(*)                           AS events
FROM github_events ge
LEFT JOIN top5 t ON ge.event_type = t.event_type
GROUP BY 1, 2
ORDER BY month, events DESC

month,event_type,events
2011-02-01T00:00:00.000Z,PushEvent,4330
2011-02-01T00:00:00.000Z,Other,2627
2011-02-01T00:00:00.000Z,WatchEvent,1133
2011-02-01T00:00:00.000Z,CreateEvent,808
2011-02-01T00:00:00.000Z,PullRequestEvent,117
2011-03-01T00:00:00.000Z,PushEvent,7320
2011-03-01T00:00:00.000Z,Other,4549
2011-03-01T00:00:00.000Z,WatchEvent,1761
2011-03-01T00:00:00.000Z,CreateEvent,1462
2011-03-01T00:00:00.000Z,PullRequestEvent,226


In [0]:
%sql
-- When does GitHub activity peak? (UTC)
SELECT
    DAYOFWEEK(created_at)     AS dow_num,     -- 1=Sun … 7=Sat
    CASE DAYOFWEEK(created_at)
        WHEN 1 THEN 'Sun' WHEN 2 THEN 'Mon' WHEN 3 THEN 'Tue'
        WHEN 4 THEN 'Wed' WHEN 5 THEN 'Thu' WHEN 6 THEN 'Fri'
        WHEN 7 THEN 'Sat'
    END                       AS day_of_week,
    HOUR(created_at)          AS hour_utc,
    COUNT(*)                  AS events
FROM github_events
GROUP BY 1, 2, 3
ORDER BY dow_num, hour_utc

dow_num,day_of_week,hour_utc,events
1,Sun,0,194843
1,Sun,1,182331
1,Sun,2,180829
1,Sun,3,190305
1,Sun,4,178871
1,Sun,5,186607
1,Sun,6,211037
1,Sun,7,209205
1,Sun,8,219447
1,Sun,9,232727


## 2. Repositories & Contributors

Who are the most active users and which repos generate the most events?

In [0]:
%sql
-- Most active repositories
SELECT
    repo_name,
    COUNT(*)                          AS total_events,
    COUNT(DISTINCT actor_login)       AS unique_contributors,
    COUNT(DISTINCT event_type)        AS event_types_seen,
    MIN(created_at)                   AS first_event,
    MAX(created_at)                   AS last_event
FROM github_events
GROUP BY repo_name
ORDER BY total_events DESC
LIMIT 25

repo_name,total_events,unique_contributors,event_types_seen,first_event,last_event
yanonono/booth-update,738030,1,1,2020-10-14T07:09:19.000Z,2022-03-11T13:43:02.000Z
Lombiq/Lombiq-Orchard-Visual-Studio-Extension,368201,4,1,2016-05-17T22:08:31.000Z,2022-02-22T18:04:31.000Z
Lombiq/NGM.Favorite,288547,1,1,2017-10-18T10:26:10.000Z,2020-06-28T15:51:00.000Z
sudouc/auto-commit,157284,3,2,2017-09-05T05:53:21.000Z,2017-09-20T07:32:29.000Z
Lombiq/Lombiq-Projections,156696,4,1,2016-09-10T03:06:22.000Z,2021-05-13T14:33:17.000Z
dotnet/roslyn,150737,74,1,2015-02-21T00:33:39.000Z,2022-03-11T18:41:46.000Z
github/gitignore,140629,135773,2,2011-02-12T14:27:57.000Z,2022-04-08T17:52:39.000Z
nyarlabo/websites,121253,1,1,2012-06-19T21:58:13.000Z,2012-08-23T04:21:17.000Z
Lombiq/Lombiq-Layouts,109777,3,1,2016-11-22T16:17:04.000Z,2022-01-06T13:17:36.000Z
cilium/cilium,107224,1214,3,2016-08-16T12:06:02.000Z,2022-03-17T23:28:43.000Z


In [0]:
%sql
-- Most active users across all event types
SELECT
    actor_login,
    COUNT(*)                          AS total_events,
    COUNT(DISTINCT repo_name)         AS repos_touched,
    COUNT(DISTINCT event_type)        AS event_types,
    MIN(created_at)                   AS first_seen,
    MAX(created_at)                   AS last_seen
FROM github_events
GROUP BY actor_login
ORDER BY total_events DESC
LIMIT 25

actor_login,total_events,repos_touched,event_types,first_seen,last_seen
dependabot[bot],2140242,242490,7,2017-05-03T09:06:39.000Z,2022-04-16T20:59:46.000Z
vercel[bot],1199036,159985,3,2020-05-27T18:06:18.000Z,2022-04-16T22:59:33.000Z
direwolf-github,1114112,399065,1,2019-03-21T14:36:20.000Z,2022-03-12T02:52:03.000Z
LombiqBot,1105716,11,2,2015-09-29T19:10:03.000Z,2020-06-28T15:51:00.000Z
github-actions[bot],774147,10220,8,2019-01-24T10:35:23.000Z,2022-04-16T22:56:50.000Z
yanonono,738089,4,1,2020-01-23T06:30:05.000Z,2022-03-11T13:43:02.000Z
dependabot-preview[bot],345000,4291,6,2019-05-23T08:15:50.000Z,2021-10-19T06:13:29.000Z
dotnet-bot,294302,35,3,2015-02-04T01:30:56.000Z,2022-03-12T02:41:41.000Z
renovate[bot],204334,4463,6,2017-09-21T17:03:05.000Z,2022-04-16T22:18:44.000Z
nmgrluser,197949,434,1,2014-06-07T20:21:05.000Z,2022-03-12T02:58:01.000Z


In [0]:
%sql
-- How are PR/issue authors associated with repos?
SELECT
    author_association,
    COUNT(*)                     AS events,
    COUNT(DISTINCT actor_login)  AS unique_users
FROM github_events
WHERE author_association IS NOT NULL
GROUP BY author_association
ORDER BY events DESC

author_association,events,unique_users
NONE,45882626,3318779
CONTRIBUTOR,1780073,76304
OWNER,1101621,99276
COLLABORATOR,664260,51014
MEMBER,571420,18347


## 3. Pull Request Analysis

Pull request lifecycle: opened → reviewed → merged/closed.

In [0]:
%sql
-- Overall PR lifecycle metrics
WITH prs AS (
    SELECT
        CONCAT(repo_name, '#', number) AS pr_id,
        MAX(CASE WHEN action = 'opened'  THEN 1 ELSE 0 END)  AS was_opened,
        MAX(CASE WHEN action = 'closed'  THEN 1 ELSE 0 END)  AS was_closed,
        MAX(CASE WHEN merged_at IS NOT NULL THEN 1 ELSE 0 END) AS was_merged,
        MAX(CASE WHEN action = 'reopened' THEN 1 ELSE 0 END) AS was_reopened,
        MIN(CASE WHEN action = 'opened' THEN created_at END)  AS opened_at,
        MIN(CASE WHEN merged_at IS NOT NULL THEN merged_at END) AS first_merged_at,
        MIN(CASE WHEN action = 'closed' THEN created_at END)  AS first_closed_at
    FROM github_events
    WHERE event_type = 'PullRequestEvent'
      AND number IS NOT NULL
    GROUP BY repo_name, number
)
SELECT
    COUNT(*)                                              AS total_prs,
    SUM(was_opened)                                       AS opened,
    SUM(was_merged)                                       AS merged,
    SUM(CASE WHEN was_closed = 1 AND was_merged = 0 THEN 1 ELSE 0 END)
                                                          AS closed_unmerged,
    ROUND(100.0 * SUM(was_merged) / NULLIF(SUM(was_opened), 0), 1)
                                                          AS merge_rate_pct,
    ROUND(AVG(
        CASE WHEN was_merged = 1
             THEN UNIX_TIMESTAMP(first_merged_at) - UNIX_TIMESTAMP(opened_at)
        END
    ) / 3600.0, 1)                                        AS avg_merge_hours,
    SUM(was_reopened)                                     AS reopened
FROM prs

total_prs,opened,merged,closed_unmerged,merge_rate_pct,avg_merge_hours,reopened
2087863,2032974,2087863,0,102.7,-438720.3,13621


In [0]:
%sql
-- Merge rate for the most PR-active repositories
WITH pr_status AS (
    SELECT
        repo_name,
        CONCAT(repo_name, '#', number) AS pr_id,
        MAX(CASE WHEN action = 'opened'  THEN 1 ELSE 0 END)  AS was_opened,
        MAX(CASE WHEN merged_at IS NOT NULL THEN 1 ELSE 0 END) AS was_merged
    FROM github_events
    WHERE event_type = 'PullRequestEvent'
      AND number IS NOT NULL
    GROUP BY repo_name, number
)
SELECT
    repo_name,
    SUM(was_opened)                                       AS prs_opened,
    SUM(was_merged)                                       AS prs_merged,
    SUM(was_opened) - SUM(was_merged)                     AS prs_unmerged,
    ROUND(100.0 * SUM(was_merged) / NULLIF(SUM(was_opened), 0), 1)
                                                          AS merge_rate_pct
FROM pr_status
GROUP BY repo_name
HAVING SUM(was_opened) >= 5
ORDER BY prs_opened DESC
LIMIT 20

repo_name,prs_opened,prs_merged,prs_unmerged,merge_rate_pct
Baystation12/Baystation12,18933,19285,-352,101.9
cilium/cilium,12506,12963,-457,103.7
BattletechModders/RogueTech,5779,5897,-118,102.0
citation-style-language/styles,5324,5448,-124,102.3
BeeStation/BeeStation-Hornet,5082,5191,-109,102.1
circleci/circleci-docs,4925,5094,-169,103.4
msys2/MINGW-packages,4242,4387,-145,103.4
SAP/fundamental-ngx,4046,4179,-133,103.3
SAP/spartacus,3792,3991,-199,105.2
citra-emu/citra,3472,3556,-84,102.4


## 4. Commit & Push Activity

Using the `commits` table for author-level commit analysis and `github_events` push metadata.

In [0]:
%sql
-- Most prolific commit authors
SELECT
    author,
    COUNT(DISTINCT commit_id)    AS unique_commits,
    COUNT(DISTINCT repo_name)    AS repos,
    AVG(push_size)               AS avg_push_size,
    MIN(created_at)              AS first_commit,
    MAX(created_at)              AS last_commit
FROM commits
GROUP BY author
ORDER BY unique_commits DESC
LIMIT 25

author,unique_commits,repos,avg_push_size,first_commit,last_commit
vercel[bot],1098004,158201,0.0,2020-05-27T18:06:18.000Z,2022-04-16T22:59:33.000Z
now[bot],63442,6307,0.0,2018-09-17T08:53:43.000Z,2020-05-22T19:08:44.000Z
github-actions[bot],30179,1707,0.0,2019-02-15T18:00:06.000Z,2022-04-16T22:56:50.000Z
palaso-bob,5052,9,0.0,2015-06-17T05:15:30.000Z,2020-11-13T14:43:41.000Z
paddle-bot[bot],4416,2,0.0,2022-03-15T03:13:03.000Z,2022-04-16T17:43:43.000Z
MarcoFalke,4148,26,0.0,2015-03-09T01:09:46.000Z,2022-04-15T17:02:58.000Z
JuliaRegistrator,4092,1098,0.0,2019-04-04T17:05:03.000Z,2022-04-16T22:44:17.000Z
deno-deploy[bot],3725,80,0.0,2021-03-29T16:26:41.000Z,2022-04-15T22:05:50.000Z
4everland[bot],3115,1789,0.0,2021-08-12T08:41:11.000Z,2022-04-16T10:29:58.000Z
streamich,2933,8,0.0,2017-12-04T09:15:57.000Z,2021-08-21T03:20:09.000Z


In [0]:
%sql
-- How many commits per push?
WITH raw_pushes AS (
    SELECT DISTINCT repo_name, created_at, push_size
    FROM commits
    WHERE push_size IS NOT NULL
),
bucketed AS (
    SELECT
        CASE
            WHEN push_size = 1          THEN '1 commit'
            WHEN push_size BETWEEN 2 AND 5   THEN '2-5 commits'
            WHEN push_size BETWEEN 6 AND 10  THEN '6-10 commits'
            WHEN push_size BETWEEN 11 AND 50 THEN '11-50 commits'
            ELSE '50+ commits'
        END AS push_bucket,
        CASE
            WHEN push_size = 1          THEN 1
            WHEN push_size BETWEEN 2 AND 5   THEN 2
            WHEN push_size BETWEEN 6 AND 10  THEN 3
            WHEN push_size BETWEEN 11 AND 50 THEN 4
            ELSE 5
        END AS sort_order
    FROM raw_pushes
)
SELECT
    push_bucket,
    COUNT(*) AS pushes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM bucketed
GROUP BY push_bucket, sort_order
ORDER BY sort_order

push_bucket,pushes,pct
50+ commits,3273863,100.0


## 5. Issues & Code Reviews

In [0]:
%sql
-- Issue lifecycle overview
SELECT
    action,
    COUNT(*)                          AS events,
    COUNT(DISTINCT actor_login)       AS unique_actors,
    COUNT(DISTINCT repo_name)         AS unique_repos
FROM github_events
WHERE event_type = 'IssuesEvent'
GROUP BY action
ORDER BY events DESC

action,events,unique_actors,unique_repos
opened,1230518,307089,115868
closed,861171,103523,62231
reopened,38218,15318,10237


In [0]:
%sql
-- Top labels attached to issues/PRs
SELECT
    label,
    COUNT(*) AS occurrences
FROM (
    SELECT EXPLODE(labels) AS label
    FROM github_events
    WHERE labels IS NOT NULL
      AND SIZE(labels) > 0
)
GROUP BY label
ORDER BY occurrences DESC
LIMIT 25

label,occurrences
dependencies,799274
enhancement,261369
bug,249353
auto-migrated,127706
javascript,75254
question,65792
:arrow_heading_down: pull,65022
Type-Defect,61491
Priority-Medium,58554
greenkeeper,41828


In [0]:
%sql
-- Pull request review outcomes
SELECT
    review_state,
    COUNT(*)                          AS reviews,
    COUNT(DISTINCT actor_login)       AS unique_reviewers,
    COUNT(DISTINCT repo_name)         AS repos
FROM github_events
WHERE event_type = 'PullRequestReviewEvent'
  AND review_state IS NOT NULL
GROUP BY review_state
ORDER BY reviews DESC

review_state,reviews,unique_reviewers,repos


## 6. File & Line-Level Changes

Using the `file_changes` and `line_changes` tables for code-change depth analysis.

In [0]:
%sql
-- Repos with the highest code churn
SELECT
    repo_name,
    COUNT(*)                   AS change_events,
    SUM(additions)             AS total_additions,
    SUM(deletions)             AS total_deletions,
    SUM(additions) - SUM(deletions) AS net_lines,
    SUM(changed_files)         AS total_files_changed,
    COUNT(DISTINCT actor_login) AS unique_authors
FROM file_changes
GROUP BY repo_name
ORDER BY change_events DESC
LIMIT 20

repo_name,change_events,total_additions,total_deletions,net_lines,total_files_changed,unique_authors
Baystation12/Baystation12,38884,39209149,28860958,10348191,637562,668
cilium/cilium,25239,56919592,21317768,35601824,544703,463
BattletechModders/RogueTech,11564,148399323,110987622,37411701,1878887,58
citation-style-language/styles,11098,8188960,1248852,6940108,180796,1399
BeeStation/BeeStation-Hornet,10687,20814387,29870906,-9056519,195580,284
circleci/circleci-docs,9839,3395215,900009,2495206,51797,806
msys2/MINGW-packages,8561,850122,1363725,-513603,29611,294
SAP/fundamental-ngx,8489,4744918,2665405,2079513,154938,87
SAP/spartacus,7508,6993902,3090832,3903070,219027,93
citra-emu/citra,6988,4220764,2188483,2032281,85832,312


In [0]:
%sql
-- What kinds of file changes are recorded?
SELECT
    change_type,
    COUNT(*)              AS events,
    ROUND(AVG(additions), 1)  AS avg_additions,
    ROUND(AVG(deletions), 1)  AS avg_deletions,
    ROUND(AVG(changed_files), 1) AS avg_files_changed
FROM file_changes
WHERE change_type IS NOT NULL
GROUP BY change_type
ORDER BY events DESC

change_type,events,avg_additions,avg_deletions,avg_files_changed
Modify,3080412,2538.6,1343.7,28.1
Add,431161,4337.8,0.0,27.2
Unknown,35198,0.0,0.0,19.0
Delete,34220,0.0,9438.6,75.5


In [0]:
%sql
-- Files that attract the most inline review comments
SELECT
    file_path,
    COUNT(*)                          AS comments,
    COUNT(DISTINCT actor_login)       AS unique_commenters,
    COUNT(DISTINCT repo_name)         AS repos
FROM line_changes
WHERE comment_body IS NOT NULL
GROUP BY file_path
ORDER BY comments DESC
LIMIT 25

file_path,comments,unique_commenters,repos
README.md,11342,3248,2354
package.json,4100,1254,886
CHANGELOG.md,3579,379,258
src/net_processing.cpp,2956,81,1
src/wallet/wallet.cpp,2872,107,2
src/wallet/rpcwallet.cpp,2461,109,2
src/validation.cpp,2284,113,4
.travis.yml,2112,696,427
src/net.cpp,1952,108,5
index.html,1860,589,516


In [0]:
%sql
-- Addition vs deletion lines across all diffs
SELECT
    line_type,
    COUNT(*)                     AS total_lines,
    COUNT(DISTINCT repo_name)    AS repos,
    COUNT(DISTINCT actor_login)  AS authors
FROM line_changes
WHERE line_type IS NOT NULL
GROUP BY line_type
ORDER BY total_lines DESC

line_type,total_lines,repos,authors
context,876966,16203,30341


## Summary & Next Steps

This exploratory analysis covered:

* **Table inventory** — row counts and temporal coverage
* **Event distribution** — event types, time-of-day patterns
* **Repos & contributors** — top actors and repositories
* **PR lifecycle** — merge rates, time-to-merge
* **Commits** — push sizes, top committers
* **Issues & reviews** — labels, review outcomes
* **File/line changes** — code churn, most-reviewed files

**Potential deep-dives:**
* Contributor retention / churn analysis
* Time-to-first-response for issues and PRs
* Repository health scoring
* Language detection from file extensions in `line_changes.file_path`